# Lab 2 – Explore raw downloads and missing values

Task 2.1: call each extractor (where available), transform the saved file and inspect with `df.head()`, `df.info()` and `df.isna().sum()`.

Indicators adapted to this project (Lab 1):
- WHO xMart: `SDGSUICIDE`
- World Bank: `SL.UEM.TOTL.ZS`, `NY.GDP.PCAP.CD`
- OWID: `life-expectancy` → `OWID_LIFE_EXPECTANCY`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from etl.extract_owid import download_owid_chart
from etl.extract_who import download_who_indicator
from etl.extract_worldbank import (
    download_worldbank_countries,
    download_worldbank_indicator,
)
from etl.transform_owid import transform_owid_chart_file
from etl.transform_who import transform_who_file
from etl.transform_worldbank import (
    transform_worldbank_country_file,
    transform_worldbank_indicator_file,
)

print("Project root:", PROJECT_ROOT)

Project root: /home/agu/Workspace/proiecte_personale/global-health-app


## 1. Extract raw files

World Bank, WHO xMart and OWID are downloaded live.

In [2]:
countries_path, countries_url = download_worldbank_countries()
print("countries:", countries_path.name)

wb_paths = {}
for code in ("SL.UEM.TOTL.ZS", "NY.GDP.PCAP.CD"):
    path, url = download_worldbank_indicator(code, years="2000:2024")
    wb_paths[code] = path
    print(code, path.name)

who_path, who_url = download_who_indicator("SDGSUICIDE")
print("WHO xMart", who_path.name)

owid_path, owid_url = download_owid_chart("life-expectancy")
print("OWID", owid_path.name)

countries: countries.json
SL.UEM.TOTL.ZS SL.UEM.TOTL.ZS_2000-2024.json
NY.GDP.PCAP.CD NY.GDP.PCAP.CD_2000-2024.json
WHO cache (skip live GHO): SDGSUICIDE_from2000.json bytes= 2020110
OWID life-expectancy_full.csv


## 2. Transform and inspect

In [4]:
def inspect(name, df):
    print("=" * 72)
    print(name, "rows=", len(df))
    display(df.head())
    df.info()
    missing = df.isna().sum()
    print("\nMissing values:\n", missing)
    if "value" in df.columns and len(df):
        share = missing["value"] / len(df)
        print(f"Share of missing values: {share:.2%}")
    return df

countries = inspect(
    "World Bank countries",
    transform_worldbank_country_file(countries_path),
)

frames = {}
for code, path in wb_paths.items():
    frames[code] = inspect(code, transform_worldbank_indicator_file(path, code))

frames["SDGSUICIDE"] = inspect(
    "SDGSUICIDE",
    transform_who_file(who_path, "SDGSUICIDE"),
)

frames["OWID_LIFE_EXPECTANCY"] = inspect(
    "OWID_LIFE_EXPECTANCY",
    transform_owid_chart_file(owid_path, "OWID_LIFE_EXPECTANCY"),
)

World Bank countries rows= 217


,iso3,iso2,name,region,income_group
0,ABW,AW,Aruba,Latin America & Caribbean,High income
1,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
2,AGO,AO,Angola,Sub-Saharan Africa,Lower middle income
3,ALB,AL,Albania,Europe & Central Asia,Upper middle income
4,AND,AD,Andorra,Europe & Central Asia,High income


<class 'pandas.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   iso3          217 non-null    str  
 1   iso2          217 non-null    str  
 2   name          217 non-null    str  
 3   region        217 non-null    str  
 4   income_group  217 non-null    str  
dtypes: str(5)
memory usage: 8.6 KB

Missing values:
 iso3            0
iso2            0
name            0
region          0
income_group    0
dtype: int64
SL.UEM.TOTL.ZS rows= 6650


,country_iso3,indicator_code,year,sex,value
0,AFE,SL.UEM.TOTL.ZS,2024,None,7.571697
1,AFE,SL.UEM.TOTL.ZS,2023,None,7.676548
2,AFE,SL.UEM.TOTL.ZS,2022,None,7.869470
3,AFE,SL.UEM.TOTL.ZS,2021,None,8.407412
4,AFE,SL.UEM.TOTL.ZS,2020,None,7.974302


<class 'pandas.DataFrame'>
RangeIndex: 6650 entries, 0 to 6649
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    6650 non-null   str    
 1   indicator_code  6650 non-null   str    
 2   year            6650 non-null   str    
 3   sex             0 non-null      object 
 4   value           5866 non-null   float64
dtypes: float64(1), object(1), str(3)
memory usage: 259.9+ KB

Missing values:
 country_iso3         0
indicator_code       0
year                 0
sex               6650
value              784
dtype: int64
Share of missing values: 11.79%
NY.GDP.PCAP.CD rows= 6650


,country_iso3,indicator_code,year,sex,value
0,AFE,NY.GDP.PCAP.CD,2024,None,1628.227289
1,AFE,NY.GDP.PCAP.CD,2023,None,1571.132704
2,AFE,NY.GDP.PCAP.CD,2022,None,1675.902524
3,AFE,NY.GDP.PCAP.CD,2021,None,1560.894626
4,AFE,NY.GDP.PCAP.CD,2020,None,1351.503167


<class 'pandas.DataFrame'>
RangeIndex: 6650 entries, 0 to 6649
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    6650 non-null   str    
 1   indicator_code  6650 non-null   str    
 2   year            6650 non-null   str    
 3   sex             0 non-null      object 
 4   value           6442 non-null   float64
dtypes: float64(1), object(1), str(3)
memory usage: 259.9+ KB

Missing values:
 country_iso3         0
indicator_code       0
year                 0
sex               6650
value              208
dtype: int64
Share of missing values: 3.13%
SDGSUICIDE rows= 12058


,country_iso3,indicator_code,year,sex,value
0,MLT,SDGSUICIDE,2005,SEX_MLE,8.218115
1,MLT,SDGSUICIDE,2005,SEX_FMLE,2.825409
2,MLI,SDGSUICIDE,2005,SEX_BTSX,3.928525
3,MLI,SDGSUICIDE,2005,SEX_MLE,5.091353
4,MLI,SDGSUICIDE,2005,SEX_FMLE,2.744844


<class 'pandas.DataFrame'>
RangeIndex: 12058 entries, 0 to 12057
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    12058 non-null  str    
 1   indicator_code  12058 non-null  str    
 2   year            12058 non-null  int64  
 3   sex             12058 non-null  str    
 4   value           12058 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 471.1 KB

Missing values:
 country_iso3      0
indicator_code    0
year              0
sex               0
value             0
dtype: int64
Share of missing values: 0.00%
OWID_LIFE_EXPECTANCY rows= 21565


,country_iso3,indicator_code,year,sex,value
0,AFG,OWID_LIFE_EXPECTANCY,1950,None,28.1563
1,AFG,OWID_LIFE_EXPECTANCY,1951,None,28.5836
2,AFG,OWID_LIFE_EXPECTANCY,1952,None,29.0138
3,AFG,OWID_LIFE_EXPECTANCY,1953,None,29.4521
4,AFG,OWID_LIFE_EXPECTANCY,1954,None,29.6975


<class 'pandas.DataFrame'>
RangeIndex: 21565 entries, 0 to 21564
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    20217 non-null  str    
 1   indicator_code  21565 non-null  str    
 2   year            21565 non-null  int64  
 3   sex             0 non-null      object 
 4   value           21565 non-null  float64
dtypes: float64(1), int64(1), object(1), str(2)
memory usage: 842.5+ KB

Missing values:
 country_iso3       1348
indicator_code        0
year                  0
sex               21565
value                 0
dtype: int64
Share of missing values: 0.00%


## 3. Missing-value summary per indicator

In [5]:
import pandas as pd

rows = []
for code, df in frames.items():
    n = len(df)
    missing_value = int(df["value"].isna().sum()) if n else 0
    rows.append({
        "indicator_code": code,
        "rows": n,
        "missing_value": missing_value,
        "missing_value_share": (missing_value / n) if n else None,
        "missing_year": int(df["year"].isna().sum()) if n else 0,
        "missing_iso3": int(df["country_iso3"].isna().sum()) if n else 0,
    })

summary = pd.DataFrame(rows)
display(summary)
print(
    "Note: World Bank returns null `value` for country-years without a measurement; "
    "those rows are dropped later in clean_observations()."
)

,indicator_code,rows,missing_value,missing_value_share,missing_year,missing_iso3
0,SL.UEM.TOTL.ZS,6650,784,0.117895,0,0
1,NY.GDP.PCAP.CD,6650,208,0.031278,0,0
2,SDGSUICIDE,12058,0,0.000000,0,0
3,OWID_LIFE_EXPECTANCY,21565,0,0.000000,0,1348


Note: World Bank returns null `value` for country-years without a measurement; those rows are dropped later in clean_observations().
